<a href="https://colab.research.google.com/github/YuriArduino/Estudos_Artificial_Intelligence/blob/Proto_Projects/Analise_de_sentimentos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install -q pandas matplotlib seaborn langchain_community langchain_google_genai faiss-cpu pymupdf requests

In [7]:
import logging
from datetime import datetime
import pytz  # biblioteca já disponível no Colab

# Define o fuso horário de Brasília (UTC-3)
brasilia_tz = pytz.timezone("America/Sao_Paulo")

class TZFormatter(logging.Formatter):
    def formatTime(self, record, datefmt=None):
        dt = datetime.fromtimestamp(record.created, tz=brasilia_tz)
        return dt.strftime(datefmt or "%H:%M:%S")

# Configuração global de logs
handler = logging.StreamHandler()
handler.setFormatter(TZFormatter("%(asctime)s | %(levelname)-7s | %(message)s"))

logging.basicConfig(
    level=logging.INFO,
    handlers=[handler],
    force=True
)

logger = logging.getLogger(__name__)
logger.info("Logging configurado com Sucesso - Horário de Brasília (UTC-3).")


00:13:09 | INFO    | Logging configurado com Sucesso - Horário de Brasília (UTC-3).


In [8]:
import logging
from pathlib import Path
from tempfile import NamedTemporaryFile
from typing import List
from urllib.parse import unquote

import requests


# Usa o logger já configurado na célula anterior
logger = logging.getLogger(__name__)


class DocumentLoader:
    """
    Carrega documentos de URLs e grava-os como arquivos temporários.
    Também ajusta os metadados 'source' usando o nome original do arquivo.
    """

    def __init__(self, timeout: int = 20):
        self.timeout = timeout

    @staticmethod
    def _extrair_nome_arquivo(url: str) -> str:
        """Extrai o nome do arquivo de uma URL, decodificando caracteres especiais."""
        return unquote(url.split("/")[-1])

    def _baixar(self, url: str) -> Path:
        """
        Baixa um arquivo de uma URL e salva como arquivo temporário.
        Retorna o caminho do arquivo criado.
        """
        dl_url = (
            url + "?raw=true"
            if "github.com" in url and "?raw=true" not in url
            else url
        )

        response = requests.get(dl_url, timeout=self.timeout)
        response.raise_for_status()

        with NamedTemporaryFile(suffix=".csv", delete=False) as tmp:
            tmp.write(response.content)
            return Path(tmp.name)

    def carregar(self, urls: List[str]) -> List[Path]:
        """
        Baixa e armazena os arquivos localmente (temporários).
        Retorna a lista de caminhos dos arquivos criados.
        """
        logger.info("Iniciando o carregamento de documentos...")
        arquivos: List[Path] = []

        for url in urls:
            nome = self._extrair_nome_arquivo(url)
            try:
                caminho = self._baixar(url)
                arquivos.append(caminho)
                logger.info(f"✅ '{nome}' carregado com sucesso")
            except Exception as e:
                logger.error(f"❌ Erro ao carregar '{nome}': {e}")

        logger.info("Processo concluído.")
        return arquivos


if __name__ == "__main__":
    urls = [
        "https://raw.githubusercontent.com/YuriArduino/Estudos_Artificial_Intelligence/refs/heads/Dados/TestReviews.csv"
    ]

    loader = DocumentLoader(timeout=20)
    arquivos_baixados = loader.carregar(urls)

    logger.info("Arquivos salvos localmente:")
    for arquivo in arquivos_baixados:
        logger.info(f"  - {arquivo}")


00:13:09 | INFO    | Iniciando o carregamento de documentos...
00:13:09 | INFO    | ✅ 'TestReviews.csv' carregado com sucesso
00:13:09 | INFO    | Processo concluído.
00:13:09 | INFO    | Arquivos salvos localmente:
00:13:09 | INFO    |   - /tmp/tmpwtbu6shj.csv


In [9]:
import os
import logging
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI

logger = logging.getLogger(__name__)

class LlmmodelLoader:
    """
    Carrega e configura um modelo LLM usando a API do Google Gemini.
    """

    def carregar_modelos(self, model: str = "gemini-2.5-flash", temperature: float = 0):
        try:
            logger.info("--- Carregando modelos ---")

            # Obtém e valida a chave da API
            api_key = userdata.get("GEMINI_API_KEY")
            if not api_key:
                raise RuntimeError("Chave GEMINI_API_KEY não encontrada no userdata do Colab.")
            os.environ["GOOGLE_API_KEY"] = api_key

            # Inicializa o modelo
            llm = ChatGoogleGenerativeAI(model=model, temperature=temperature)

            logger.info(f"✅ Modelo '{model}' configurado com sucesso!")
            return llm

        except Exception as e:
            logger.error(f"❌ Erro na configuração do modelo: {e}")
            return None


if __name__ == "__main__":
    loader = LlmmodelLoader()
    llm_model = loader.carregar_modelos()

    if llm_model:
        logger.info("Modelo LLM carregado com sucesso.")
    else:
        logger.error("Falha ao carregar o modelo LLM.")


00:13:11 | INFO    | --- Carregando modelos ---
00:13:11 | INFO    | ✅ Modelo 'gemini-2.5-flash' configurado com sucesso!
00:13:11 | INFO    | Modelo LLM carregado com sucesso.


In [10]:
import pandas as pd

# Load the CSV file into a pandas DataFrame
csv_file_path = '/tmp/tmp4euod22f.csv'
try:
    df_loaded_csv = pd.read_csv(csv_file_path)
    df_reviews = df_loaded_csv.copy()
    print(f"Successfully loaded CSV from: {csv_file_path}")
    # Display the first few rows of the DataFrame
    display(df_reviews.head())
except FileNotFoundError:
    print(f"Error: The file {df_reviews} was not found.")
except Exception as e:
    print(f"An error occurred while loading the CSV: {e}")

00:13:13 | INFO    | NumExpr defaulting to 2 threads.


Successfully loaded CSV from: /tmp/tmp4euod22f.csv


,review,class
0,Fantastic spot for an even or a quite cocktail...,1
1,"Love, love, love the calamari. It's so good an...",1
2,"Love this place. Stiff martinis and cocktails,...",1
3,It's everything a great cocktail bar should be...,1
4,"I came here before a pirates game, so it was a...",1


In [11]:
df_reviews["class"].value_counts()

,count
class,
1,2989
0,1332


In [14]:
# Extract the 'review' column
coluna_de_reviews = df_reviews["review"]

# Select 40 random reviews and save them to a new variable
test_reviews = coluna_de_reviews.sample(n=40, random_state=42) # Using a random_state for reproducibility


print("First 5 random reviews selected for test_reviews:")
display(test_reviews.head())

print(f"\nTotal number of reviews in test_reviews: {len(test_reviews)}")

First 5 random reviews selected for test_reviews:


,review
1073,Awesome! It is sooo much fun for me to take p...
856,The Rhythm Room was the first blues bar I visi...
1222,"As a former BR employee, I'm very hard on my s..."
3406,"Sadly, the professionalism of the front desk n..."
2250,4.5 .5 off for hard ass tires.This is awesome ...



Total number of reviews in test_reviews: 40


In [17]:
import time # Import the time module
import pandas as pd # Ensure pandas is imported for DataFrame creation

lista_de_analises_de_sentimentos = []
total_reviews = len(test_reviews) # Use test_reviews as decided in previous steps
llm_call_count = 0 # Initialize a counter for LLM calls

logger.info(f"Iniciando análise de sentimento para {total_reviews} resenhas...")

for review_numero, resenha in enumerate(test_reviews, start=1): # Iterate over test_reviews
    prompt = f"""
    Analise o sentimento da seguinte resenha de forma direta.
    Responda APENAS com uma das palavras:
    'Positiva', 'Negativa' ou 'Neutra'.

    Exemplos:
    "Eu adorei esse produto" -> Positiva
    "Gostei, mas não é nada de especial" -> Neutra
    "Odiei esse produto" -> Negativa

    Resenha: "{resenha}"
    """

    try:
        # Make the LLM call
        resposta = llm_model.invoke(prompt) # Using .invoke() which is standard in newer LangChain
        llm_call_count += 1 # Increment the counter after a successful or attempted call

        # Extract sentiment from the response
        if hasattr(resposta, "content"):
            sentimento = resposta.content.strip()
        else:
            # Fallback for older LangChain versions or different response types
            sentimento = str(resposta).strip()

        # Validate the response against expected sentiments
        if sentimento not in ["Positiva", "Negativa", "Neutra"]:
            # Log a warning if the sentiment is unexpected
            logger.warning(f"Resenha {review_numero}/{total_reviews}: Resposta inesperada do LLM: '{sentimento}'. Definindo como Neutra.")
            sentimento = "Neutra" # fallback if response is not one of the three expected

    except Exception as e:
        # Log the error with review number and a snippet of the review
        logger.error(f"❌ Erro ao analisar a resenha {review_numero}/{total_reviews} ('{resenha[:50]}...'): {e}")
        sentimento = "Neutra" # fallback in case of error

    lista_de_analises_de_sentimentos.append(sentimento)

    # Add a delay every 5 calls
    if llm_call_count % 5 == 0:
        logger.info(f"Pausa de 14 segundos para evitar exceder o limite de recursos...")
        time.sleep(14)


    # Log progress periodically, e.g., every 10 reviews or at the end
    if review_numero % 10 == 0 or review_numero == total_reviews:
        logger.info(f"Progresso: {review_numero}/{total_reviews} resenhas processadas.")


logger.info("Análise de sentimento concluída.")

# Create a DataFrame to display test reviews and their sentiments
# Ensure the sentiment results are aligned with the test_reviews index
sentiment_results_series = pd.Series(lista_de_analises_de_sentimentos, index=test_reviews.index)

# Create a new DataFrame from the test_reviews Series and the sentiment Series
test_reviews_with_sentiment = pd.DataFrame({
    "Review": test_reviews,
    "Sentiment": sentiment_results_series
})

# Display the table
print("\nAnálises de Sentimentos para as resenhas de teste:")
display(test_reviews_with_sentiment)

00:24:09 | INFO    | Iniciando análise de sentimento para 40 resenhas...
00:24:27 | INFO    | Pausa de 14 segundos para evitar exceder o limite de recursos...
00:24:58 | INFO    | Pausa de 14 segundos para evitar exceder o limite de recursos...
00:25:12 | INFO    | Progresso: 10/40 resenhas processadas.
00:25:31 | INFO    | Pausa de 14 segundos para evitar exceder o limite de recursos...
00:25:58 | INFO    | Pausa de 14 segundos para evitar exceder o limite de recursos...
00:26:12 | INFO    | Progresso: 20/40 resenhas processadas.
00:26:27 | INFO    | Pausa de 14 segundos para evitar exceder o limite de recursos...
00:27:02 | INFO    | Pausa de 14 segundos para evitar exceder o limite de recursos...
00:27:16 | INFO    | Progresso: 30/40 resenhas processadas.
00:27:33 | INFO    | Pausa de 14 segundos para evitar exceder o limite de recursos...
00:28:04 | INFO    | Pausa de 14 segundos para evitar exceder o limite de recursos...
00:28:18 | INFO    | Progresso: 40/40 resenhas processadas.


Análises de Sentimentos para as resenhas de teste:


,Review,Sentiment
1073,Awesome! It is sooo much fun for me to take p...,Positiva
856,The Rhythm Room was the first blues bar I visi...,Positiva
1222,"As a former BR employee, I'm very hard on my s...",Positiva
3406,"Sadly, the professionalism of the front desk n...",Negativa
2250,4.5 .5 off for hard ass tires.This is awesome ...,Positiva
3559,"Just to clarify, my rating for this business i...",Negativa
17,I've been waiting to change from four stars to...,Positiva
1653,In the last 6 months Ricks has definitely had ...,Positiva
1391,"Oh my, I love Sir Ed's. It is without a doubt ...",Positiva
2083,This is my favorite place to visit for some de...,Positiva
